# Data Pack Export for Student Diploma

Exports a self-contained JSON data pack for the GED visualization web app.

**Prerequisites**: Docker services running (Neo4j, Kafka, Nuclio functions deployed).

**Output**: `data_pack/` directory with concepts, images, and GED results.

In [ ]:
import copy
import importlib
import json
import os
import random
import sys
import time
import types
import uuid
from pathlib import Path
from shutil import copyfile
from typing import Any, Dict, List, Optional, Tuple

import networkx as nx
import numpy as np
from neo4j import GraphDatabase

_src_training = os.path.dirname(os.path.abspath("data_pack_export.ipynb"))
_src = os.path.dirname(_src_training)
for _p in [_src_training, _src]:
    if _p not in sys.path:
        sys.path.insert(0, _p)

_classification = os.path.join(_src, "classification")
if _classification not in sys.path:
    sys.path.insert(0, _classification)

# Stub 'ot' (POT library) so graph_similarity/__init__.py can import FGWComparator without crashing.
# We only need GED, not FGW.
if "ot" not in sys.modules:
    sys.modules["ot"] = types.ModuleType("ot")

from classification.graph_similarity.cost_functions import (
    node_subst_cost, node_del_cost, node_ins_cost,
    edge_match, edge_del_cost, edge_ins_cost,
    features, NodeCost,
    _calculate_property_similarity_cost,
    _resolve_weight,
)
from classification.graph_similarity.node_pair_cost_kernel import build_cost_matrix
from classification.services.graph_complexity_service import GraphComplexityService
from classification.repository.concept_repository import ConceptRepository
from classification.repository.image_repository import ImageRepository

import classifier
importlib.reload(classifier)
from classifier import classify_images_stream, extract_class_from_concept_id

# --- Configuration ---
NEO4J_URI = os.environ.get("NEO4J_DSN", "bolt://localhost:7687")
NEO4J_USER = os.environ.get("NEO4J_USER", "neo4j")
NEO4J_PASSWORD = os.environ.get("NEO4J_PASSWORD", "111122223333")

OUTPUT_DIR = Path("../../data_pack")
IMAGES_PER_CLASS = 10
GED_TIMEOUT = 10.0
CLASSES = list(range(10))

LOCAL_TEST_PATH = "../../datasets/test/{cls}"
NUCLIO_TEST_PATH = "/opt/nuclio/shared_storage/test/{cls}"

print(f"Output: {OUTPUT_DIR.resolve()}")
print(f"Classes: {CLASSES}")
print(f"Images per class: {IMAGES_PER_CLASS}")
print(f"GED timeout: {GED_TIMEOUT}s")

## 1. Export Concept Graphs from Neo4j

In [2]:
def serialize_graph_node(node_id, node_data: dict) -> dict:
    """Convert a NetworkX node to a JSON-serializable dict."""
    labels = node_data.get("labels", set())
    if isinstance(labels, set):
        labels = sorted(labels)

    result = {"id": str(node_id), "labels": labels}
    skip = {"labels", "is_concept", "id", "image_id", "concept_id", "session_id"}
    for k, v in node_data.items():
        if k in skip:
            continue
        if isinstance(v, set):
            v = sorted(v)
        if isinstance(v, float) and (np.isnan(v) or np.isinf(v)):
            v = None
        result[k] = v
    return result


def serialize_graph(G: nx.Graph) -> dict:
    """Convert a NetworkX graph to the data pack JSON format."""
    nodes = [serialize_graph_node(n, d) for n, d in G.nodes(data=True)]
    edges = [{"source": str(u), "target": str(v)} for u, v in G.edges()]
    return {"nodes": nodes, "edges": edges}


def write_json(data: Any, path: Path):
    path.parent.mkdir(parents=True, exist_ok=True)
    with open(path, "w") as f:
        json.dump(data, f, indent=2, default=str)


# Connect to Neo4j
driver = GraphDatabase.driver(NEO4J_URI, auth=(NEO4J_USER, NEO4J_PASSWORD))
driver.verify_connectivity()

concept_repo = ConceptRepository(driver)
image_repo = ImageRepository(driver)
complexity_service = GraphComplexityService()

# Load concept graphs
concept_ids = concept_repo.get_all_concept_ids()
print(f"Found {len(concept_ids)} concepts: {sorted(concept_ids)}")

concept_graphs: Dict[str, nx.Graph] = {}
for cid in sorted(concept_ids):
    concept_graphs[cid] = concept_repo.get_concept_graph(cid)

# Export concepts
concepts_dir = OUTPUT_DIR / "concepts"
concepts_index = []

for cid, G in concept_graphs.items():
    digit_class = int(cid.split("_")[0])
    complexity = complexity_service.get_default_graph_complexity(G)

    concepts_index.append({
        "concept_id": cid,
        "digit_class": digit_class,
        "complexity": complexity,
        "node_count": G.number_of_nodes(),
        "edge_count": G.number_of_edges(),
    })

    graph_data = serialize_graph(G)
    graph_data["concept_id"] = cid
    write_json(graph_data, concepts_dir / f"{cid}.json")

write_json(concepts_index, OUTPUT_DIR / "concepts.json")
print(f"\nExported {len(concepts_index)} concepts:")
for c in concepts_index:
    print(f"  {c['concept_id']}: {c['node_count']} nodes, {c['edge_count']} edges, complexity={c['complexity']}")

Received notification from DBMS server: <GqlStatusObject gql_status='01N00', status_description='warn: feature deprecated. CALL subquery without a variable scope clause is deprecated. Use CALL () { ... }', position=<SummaryInputPosition line=2, column=13, offset=13>, raw_classification='DEPRECATION', classification=<NotificationClassification.DEPRECATION: 'DEPRECATION'>, raw_severity='WARNING', severity=<NotificationSeverity.WARNING: 'WARNING'>, diagnostic_record={'_classification': 'DEPRECATION', '_severity': 'WARNING', '_position': {'offset': 13, 'line': 2, 'column': 13}, 'OPERATION': '', 'OPERATION_CODE': '0', 'CURRENT_SCHEMA': '/'}> for query: '\n            CALL {\n                MATCH (n:Point {concept_id: $concept_id}) RETURN n\n                UNION ALL\n                MATCH (n:Vector {concept_id: $concept_id}) RETURN n\n                UNION ALL\n                MATCH (n:StartPoint {concept_id: $concept_id}) RETURN n\n            }\n            WITH n, labels(n) AS node_labe

Found 13 concepts: ['0_1', '1_1', '1_3', '2_1', '2_2', '3_1', '4_1', '4_2', '5_1', '6_1', '7_1', '8_1', '9_2']

Exported 13 concepts:
  0_1: 10 nodes, 10 edges, complexity=20
  1_1: 7 nodes, 6 edges, complexity=13
  1_3: 3 nodes, 2 edges, complexity=5
  2_1: 7 nodes, 6 edges, complexity=13
  2_2: 12 nodes, 12 edges, complexity=24
  3_1: 11 nodes, 10 edges, complexity=21
  4_1: 8 nodes, 8 edges, complexity=16
  4_2: 7 nodes, 6 edges, complexity=13
  5_1: 7 nodes, 6 edges, complexity=13
  6_1: 10 nodes, 10 edges, complexity=20
  7_1: 5 nodes, 4 edges, complexity=9
  8_1: 13 nodes, 14 edges, complexity=27
  9_2: 8 nodes, 8 edges, complexity=16


## 2. Process Images Through Pipeline

Select test images, submit through the full pipeline (connector → skel → contour → classification) with `delete_image_nodes=False` so image graphs remain in Neo4j.

In [3]:
from tqdm.notebook import tqdm

# Select random test images
all_images: List[Tuple[str, str, str, Dict[str, Any]]] = []  # (nuclio_path, image_id, class, params)
id_to_local_path: Dict[str, str] = {}

for cls in CLASSES:
    local_folder = LOCAL_TEST_PATH.format(cls=cls)
    nuclio_folder = NUCLIO_TEST_PATH.format(cls=cls)
    images = sorted(f for f in os.listdir(local_folder) if f.endswith(".png"))
    selected = random.sample(images, min(IMAGES_PER_CLASS, len(images)))

    for fname in selected:
        image_id = str(uuid.uuid4())
        params = {
            "image_id": image_id,
            "delete_image_nodes": False,
            "ged_timeout": GED_TIMEOUT,
            "disable_tracing": True,
        }
        all_images.append((os.path.join(nuclio_folder, fname), image_id, str(cls), params))
        id_to_local_path[image_id] = os.path.join(local_folder, fname)

print(f"Selected {len(all_images)} images ({IMAGES_PER_CLASS} per class)")
print(f"Submitting through pipeline...")

# Submit and collect classification results
id_to_expected = {img_id: expected for _, img_id, expected, _ in all_images}
pbar = tqdm(total=len(all_images), desc="Processing")

stream_input = [(path, img_id, img_params) for path, img_id, _, img_params in all_images]
stream_results = classify_images_stream(
    stream_input,
    on_result=lambda img_id, _r: pbar.update(1),
    idle_timeout=300,
)
pbar.close()

# Summarize results
success = sum(1 for r in stream_results.values() if r["status"] == "success")
errors = sum(1 for r in stream_results.values() if r["status"] != "success")
print(f"\nPipeline complete: {success} success, {errors} errors")

# Determine correct/incorrect for each image
for img_id, result in stream_results.items():
    expected = id_to_expected[img_id]
    result["expected_class"] = expected
    if result["status"] == "success":
        class_results = result.get("classification_results", [])
        if class_results and class_results[0].get("is_minor", False):
            predicted = extract_class_from_concept_id(class_results[0]["concept_id"])
            result["predicted_class"] = predicted
            result["correct"] = predicted == expected
        else:
            result["predicted_class"] = "unclassified"
            result["correct"] = False
    else:
        result["predicted_class"] = "error"
        result["correct"] = False

correct_count = sum(1 for r in stream_results.values() if r.get("correct"))
print(f"Correct: {correct_count}/{success} ({correct_count/max(success,1)*100:.1f}%)")

Selected 100 images (10 per class)
Submitting through pipeline...


Processing:   0%|          | 0/100 [00:00<?, ?it/s]


Pipeline complete: 100 success, 0 errors
Correct: 69/100 (69.0%)


## 3. Export Image Graphs + PNGs

Query each image graph from Neo4j (still present because `delete_image_nodes=False`) and copy the original PNG.

In [4]:
# Export image graphs and PNGs
images_dir = OUTPUT_DIR / "images"
images_index = []
image_graphs: Dict[str, nx.Graph] = {}
skipped = []

successful_ids = [
    img_id for img_id, r in stream_results.items() if r["status"] == "success"
]

for img_id in tqdm(successful_ids, desc="Exporting image graphs"):
    result = stream_results[img_id]

    G = image_repo.get_image_graph(img_id)
    if G.number_of_nodes() == 0:
        skipped.append(img_id)
        continue

    image_graphs[img_id] = G
    complexity = complexity_service.get_default_graph_complexity(G)

    # Find best similarity from pipeline results
    class_results = result.get("classification_results", [])
    best_similarity = class_results[0]["similarity"] if class_results else 0.0

    images_index.append({
        "image_id": img_id,
        "digit_class": int(result["expected_class"]),
        "image_path": f"images/{img_id}.png",
        "complexity": complexity,
        "node_count": G.number_of_nodes(),
        "edge_count": G.number_of_edges(),
        "predicted_class": int(result["predicted_class"]) if result["predicted_class"].isdigit() else -1,
        "best_similarity": round(best_similarity, 4),
        "correct": result.get("correct", False),
    })

    # Export graph JSON
    graph_data = serialize_graph(G)
    graph_data["image_id"] = img_id
    write_json(graph_data, images_dir / f"{img_id}.json")

    # Copy PNG
    local_path = id_to_local_path[img_id]
    dst_png = images_dir / f"{img_id}.png"
    dst_png.parent.mkdir(parents=True, exist_ok=True)
    copyfile(local_path, dst_png)

write_json(images_index, OUTPUT_DIR / "images.json")

if skipped:
    print(f"WARNING: {len(skipped)} images had empty graphs (skipped)")
print(f"Exported {len(images_index)} image graphs + PNGs")
print(f"  Correct: {sum(1 for i in images_index if i['correct'])}")
print(f"  Incorrect: {sum(1 for i in images_index if not i['correct'])}")

Exporting image graphs:   0%|          | 0/100 [00:00<?, ?it/s]

Received notification from DBMS server: <GqlStatusObject gql_status='01N00', status_description='warn: feature deprecated. CALL subquery without a variable scope clause is deprecated. Use CALL () { ... }', position=<SummaryInputPosition line=2, column=13, offset=13>, raw_classification='DEPRECATION', classification=<NotificationClassification.DEPRECATION: 'DEPRECATION'>, raw_severity='WARNING', severity=<NotificationSeverity.WARNING: 'WARNING'>, diagnostic_record={'_classification': 'DEPRECATION', '_severity': 'WARNING', '_position': {'offset': 13, 'line': 2, 'column': 13}, 'OPERATION': '', 'OPERATION_CODE': '0', 'CURRENT_SCHEMA': '/'}> for query: '\n            CALL {\n                MATCH (n:Point {image_id: $image_id}) RETURN n\n                UNION ALL\n                MATCH (n:Vector {image_id: $image_id}) RETURN n\n            }\n            WITH n, labels(n) as node_labels, properties(n) as node_props\n            OPTIONAL MATCH (n)-[r]-(m {image_id: $image_id})\n            W

Exported 100 image graphs + PNGs
  Correct: 69
  Incorrect: 31


## 4. Compute GED with Edit Paths

For each image × eligible concept pair:
1. Apply the same preprocessing as the production classifier (StartPoint + CriticalPoint)
2. Run `nx.optimize_edit_paths` to get node/edge edit operations (not just costs)
3. Build cost matrix via `node_pair_cost_kernel`
4. Compute per-property cost breakdown for substitution operations
5. Serialize to `ged_results/{image_id}/{concept_id}.json`

In [ ]:
def compute_per_property_costs(image_node_data: dict, concept_node_data: dict) -> dict:
    """Compute individual property costs for a substitution, mirroring _calculate_properties_similarity_cost.

    Weights come from `_resolve_weight`, which combines scale-strength (Parzhyn Sec. 6.1)
    with diagnostic weighting (1 / (range_width + epsilon)).
    """
    property_costs = {}
    common_properties = set(concept_node_data.keys()) & set(image_node_data.keys())
    common_properties_to_check = common_properties.intersection(features)
    if not common_properties_to_check:
        return property_costs

    raw_weights = {p: _resolve_weight(p, concept_node_data) for p in common_properties_to_check}
    total_weight = sum(raw_weights.values())
    if total_weight < 1e-9:
        return property_costs

    for feat in features:
        c_val = concept_node_data.get(feat)
        i_val = image_node_data.get(feat)

        if c_val is None and i_val is None:
            continue
        if i_val is None and c_val is not None:
            continue

        w = raw_weights.get(feat)
        if w is None:
            continue
        nw = w / total_weight

        if c_val is None and i_val is not None:
            property_costs[feat] = round(nw, 6)
            continue

        cost = _calculate_property_similarity_cost(c_val, i_val, feat, nw)
        property_costs[feat] = round(min(cost, nw), 6)

    return property_costs


def extract_edit_operations(
    node_path: list,
    edge_path: list,
    image_graph: nx.Graph,
    concept_graph: nx.Graph,
) -> Tuple[List[dict], List[dict]]:
    """Decode nx.optimize_edit_paths output into structured operations."""
    node_ops = []
    for node1, node2 in node_path:
        if node1 is None:
            cost = node_ins_cost(concept_graph.nodes[node2])
            node_ops.append({
                "type": "insertion",
                "image_node": None,
                "concept_node": str(node2),
                "cost": round(float(cost), 6),
            })
        elif node2 is None:
            cost = node_del_cost(image_graph.nodes[node1])
            node_ops.append({
                "type": "deletion",
                "image_node": str(node1),
                "concept_node": None,
                "cost": round(float(cost), 6),
            })
        else:
            cost = node_subst_cost(image_graph.nodes[node1], concept_graph.nodes[node2])
            prop_costs = compute_per_property_costs(
                image_graph.nodes[node1], concept_graph.nodes[node2]
            )
            node_ops.append({
                "type": "substitution",
                "image_node": str(node1),
                "concept_node": str(node2),
                "cost": round(float(cost), 6),
                "property_costs": prop_costs,
            })

    edge_ops = []
    for edge1, edge2 in edge_path:
        if edge1 is None:
            cost = edge_ins_cost(concept_graph.edges[edge2])
            edge_ops.append({
                "type": "insertion",
                "image_edge": None,
                "concept_edge": [str(edge2[0]), str(edge2[1])],
                "cost": round(float(cost), 6),
            })
        elif edge2 is None:
            cost = edge_del_cost(image_graph.edges[edge1])
            edge_ops.append({
                "type": "deletion",
                "image_edge": [str(edge1[0]), str(edge1[1])],
                "concept_edge": None,
                "cost": round(float(cost), 6),
            })
        else:
            matched = edge_match(image_graph.edges[edge1], concept_graph.edges[edge2])
            if matched:
                cost = 0.0
            else:
                cost = edge_del_cost(image_graph.edges[edge1]) + edge_ins_cost(concept_graph.edges[edge2])
            edge_ops.append({
                "type": "substitution",
                "image_edge": [str(edge1[0]), str(edge1[1])],
                "concept_edge": [str(edge2[0]), str(edge2[1])],
                "cost": round(float(cost), 6),
            })

    return node_ops, edge_ops


def compute_ged_with_edit_paths(
    image_graph: nx.Graph,
    concept_graph: nx.Graph,
    timeout: float,
) -> Optional[Tuple[float, List[dict], List[dict]]]:
    """
    Run nx.optimize_edit_paths with timeout, return (cost, node_ops, edge_ops).
    Returns None if no path found within timeout.
    """
    deadline = time.monotonic() + timeout
    best_result = None

    try:
        for node_path, edge_path, cost in nx.optimize_edit_paths(
            image_graph,
            concept_graph,
            node_subst_cost=node_subst_cost,
            node_del_cost=node_del_cost,
            node_ins_cost=node_ins_cost,
            edge_match=edge_match,
            edge_del_cost=edge_del_cost,
            edge_ins_cost=edge_ins_cost,
        ):
            node_ops, edge_ops = extract_edit_operations(
                node_path, edge_path, image_graph, concept_graph
            )
            best_result = (cost, node_ops, edge_ops)

            if time.monotonic() >= deadline:
                break
    except Exception as e:
        print(f"  GED error: {e}")

    return best_result


print("GED helper functions defined.")

In [6]:
from services.pre_processing.start_point_preprocessor import StartPointPreprocessor
from services.pre_processing.critical_point_preprocessor import CriticalPointPreprocessor
from services.graph_analyzer import GraphAnalyzer
from reduction_strategy.endpoint_strategy import EndpointReductionStrategy
from reduction_strategy.intersection_strategy import IntersectionPointReductionStrategy
from reduction_strategy.corner_point_reduction_strategy import CornerPointReductionStrategy
from node_similarity_calculator import NodeSimilarityCalculator
from common.traversal.visitors import AngleVisitor, QuadrantVisitor, DirectionVisitor

start_preprocessor = StartPointPreprocessor()
critical_preprocessor = CriticalPointPreprocessor(
    endpoint_reduction_strategy=EndpointReductionStrategy(
        node_similarity_calculator=NodeSimilarityCalculator()
    ),
    intersection_reduction_strategy=IntersectionPointReductionStrategy(
        node_similarity_calculator=NodeSimilarityCalculator()
    ),
    corner_point_reduction_strategy=CornerPointReductionStrategy(
        node_similarity_calculator=NodeSimilarityCalculator()
    ),
)

ged_results_dir = OUTPUT_DIR / "ged_results"
total_pairs = len(image_graphs) * len(concept_graphs)
ged_count = 0
skip_count = 0

pbar = tqdm(total=len(image_graphs), desc="GED computation (images)")

for img_id, raw_image_graph in image_graphs.items():
    image_complexity = complexity_service.get_default_graph_complexity(raw_image_graph)

    for cid, raw_concept_graph in concept_graphs.items():
        concept_complexity = complexity_service.get_default_graph_complexity(raw_concept_graph)

        # Same pre-filter as production: skip if concept more complex than image
        if concept_complexity > image_complexity:
            # Still create a result entry so the student can see why it was skipped
            skip_result = {
                "image_id": img_id,
                "concept_id": cid,
                "skipped": True,
                "reason": f"concept_complexity ({concept_complexity}) > image_complexity ({image_complexity})",
                "similarity": 0.0,
            }
            write_json(skip_result, ged_results_dir / img_id / f"{cid}.json")
            skip_count += 1
            continue

        # Preprocessing: same as ConceptMinorClassifier.check_single_concept
        image_graph_copy = copy.deepcopy(raw_image_graph)
        try:
            image_graph_copy = start_preprocessor.preprocess(
                inference_graph=image_graph_copy,
                concept_graph=raw_concept_graph,
            )
            GraphAnalyzer(
                graph=image_graph_copy,
                visitors=[
                    AngleVisitor(image_graph_copy),
                    QuadrantVisitor(image_graph_copy),
                    DirectionVisitor(image_graph_copy),
                ],
            ).analyze()
            prep_image, prep_concept = critical_preprocessor.preprocess_graphs(
                inference_graph=image_graph_copy,
                concept_graph=raw_concept_graph,
            )
        except Exception as e:
            error_result = {
                "image_id": img_id,
                "concept_id": cid,
                "skipped": True,
                "reason": f"preprocessing failed: {e}",
                "similarity": 0.0,
            }
            write_json(error_result, ged_results_dir / img_id / f"{cid}.json")
            skip_count += 1
            continue

        # Compute GED with edit paths
        ged_result = compute_ged_with_edit_paths(prep_image, prep_concept, GED_TIMEOUT)
        if ged_result is None:
            error_result = {
                "image_id": img_id,
                "concept_id": cid,
                "skipped": True,
                "reason": "GED computation returned no result",
                "similarity": 0.0,
            }
            write_json(error_result, ged_results_dir / img_id / f"{cid}.json")
            skip_count += 1
            continue

        raw_cost, node_ops, edge_ops = ged_result

        # Boria normalization
        n1 = prep_image.number_of_nodes() + prep_image.number_of_edges()
        n2 = prep_concept.number_of_nodes() + prep_concept.number_of_edges()
        similarity = 1.0 - (raw_cost / (raw_cost + max(n1, n2, 1)))

        # Build cost matrix on preprocessed graphs
        M, con_nodes, img_nodes = build_cost_matrix(prep_image, prep_concept)

        result_data = {
            "image_id": img_id,
            "concept_id": cid,
            "skipped": False,
            "similarity": round(similarity, 4),
            "raw_cost": round(float(raw_cost), 4),
            "n1": n1,
            "n2": n2,
            "node_operations": node_ops,
            "edge_operations": edge_ops,
            "cost_matrix": {
                "concept_node_ids": [str(n) for n in con_nodes],
                "image_node_ids": [str(n) for n in img_nodes],
                "matrix": np.round(M, 4).tolist(),
            },
        }
        write_json(result_data, ged_results_dir / img_id / f"{cid}.json")
        ged_count += 1

    pbar.update(1)

pbar.close()
print(f"\nGED complete: {ged_count} computed, {skip_count} skipped")

GED computation (images):   0%|          | 0/100 [00:00<?, ?it/s]

No segments with paths found
No segments with paths found
No segments with paths found
No segments with paths found
No segments with paths found
No segments with paths found
No segments with paths found
No segments with paths found
No segments with paths found
No segments with paths found
No segments with paths found
No segments with paths found
No segments with paths found
No segments with paths found
No segments with paths found
No segments with paths found
No segments with paths found
No segments with paths found
No segments with paths found
No segments with paths found
No segments with paths found
No segments with paths found
No segments with paths found
No segments with paths found
No segments with paths found
No segments with paths found
No segments with paths found
No segments with paths found
No segments with paths found
No segments with paths found
No segments with paths found
No segments with paths found
No segments with paths found
No segments with paths found
No segments wi


GED complete: 353 computed, 947 skipped


## 5. Cleanup & Verification

Remove exported image nodes from Neo4j (optional) and verify the data pack structure.

In [7]:
# Clean up image nodes from Neo4j
print("Removing exported image nodes from Neo4j...")
for img_id in tqdm(list(image_graphs.keys()), desc="Cleanup"):
    image_repo.remove_image_nodes(img_id)

driver.close()
print("Neo4j cleanup complete.")

Removing exported image nodes from Neo4j...


Cleanup:   0%|          | 0/100 [00:00<?, ?it/s]

Received notification from DBMS server: <GqlStatusObject gql_status='01N00', status_description='warn: feature deprecated. CALL subquery without a variable scope clause is deprecated. Use CALL () { ... }', position=<SummaryInputPosition line=2, column=13, offset=13>, raw_classification='DEPRECATION', classification=<NotificationClassification.DEPRECATION: 'DEPRECATION'>, raw_severity='WARNING', severity=<NotificationSeverity.WARNING: 'WARNING'>, diagnostic_record={'_classification': 'DEPRECATION', '_severity': 'WARNING', '_position': {'offset': 13, 'line': 2, 'column': 13}, 'OPERATION': '', 'OPERATION_CODE': '0', 'CURRENT_SCHEMA': '/'}> for query: '\n            CALL {\n                MATCH (n:Point {image_id: $image_id})\n                DETACH DELETE n\n            }\n            CALL {\n                MATCH (n:Vector {image_id: $image_id})\n                DETACH DELETE n\n            }\n        '
Received notification from DBMS server: <GqlStatusObject gql_status='01N00', status_

Neo4j cleanup complete.


In [8]:
# Verify data pack structure
print(f"Data pack location: {OUTPUT_DIR.resolve()}\n")

concepts_file = OUTPUT_DIR / "concepts.json"
images_file = OUTPUT_DIR / "images.json"

with open(concepts_file) as f:
    concepts = json.load(f)
with open(images_file) as f:
    images = json.load(f)

# Count GED result files
ged_dir = OUTPUT_DIR / "ged_results"
ged_files = list(ged_dir.rglob("*.json"))
ged_computed = 0
ged_skipped = 0
for gf in ged_files:
    with open(gf) as f:
        r = json.load(f)
    if r.get("skipped"):
        ged_skipped += 1
    else:
        ged_computed += 1

print(f"concepts.json: {len(concepts)} concepts")
print(f"images.json:   {len(images)} images")
print(f"GED results:   {ged_computed} computed + {ged_skipped} skipped = {len(ged_files)} total")
print(f"Image PNGs:    {len(list((OUTPUT_DIR / 'images').glob('*.png')))} files")
print(f"Image JSONs:   {len(list((OUTPUT_DIR / 'images').glob('*.json')))} files")
print(f"Concept JSONs: {len(list((OUTPUT_DIR / 'concepts').glob('*.json')))} files")

# Sample one GED result
sample_img = images[0]
sample_ged_path = ged_dir / sample_img["image_id"]
sample_ged_files = sorted(sample_ged_path.glob("*.json"))
if sample_ged_files:
    with open(sample_ged_files[0]) as f:
        sample = json.load(f)
    print(f"\n--- Sample GED result: image={sample['image_id'][:8]}... × concept={sample['concept_id']} ---")
    if not sample.get("skipped"):
        print(f"  Similarity: {sample['similarity']}")
        print(f"  Raw cost:   {sample['raw_cost']}")
        print(f"  Node ops:   {len(sample['node_operations'])} (sub={sum(1 for o in sample['node_operations'] if o['type']=='substitution')}, del={sum(1 for o in sample['node_operations'] if o['type']=='deletion')}, ins={sum(1 for o in sample['node_operations'] if o['type']=='insertion')})")
        print(f"  Edge ops:   {len(sample['edge_operations'])}")
        print(f"  Cost matrix: {len(sample['cost_matrix']['concept_node_ids'])}×{len(sample['cost_matrix']['image_node_ids'])}")
    else:
        print(f"  Skipped: {sample['reason']}")

print("\n✓ Data pack export complete!")

Data pack location: /Users/mlapin/Development/personal/NaturalAGI/data_pack

concepts.json: 13 concepts
images.json:   100 images
GED results:   353 computed + 947 skipped = 1300 total
Image PNGs:    100 files
Image JSONs:   100 files
Concept JSONs: 13 files

--- Sample GED result: image=912ee5ae... × concept=0_1 ---
  Skipped: concept_complexity (20) > image_complexity (5)

✓ Data pack export complete!
